In [ ]:
%pip install -U langchain-community

In [ ]:
%pip install -U arxiv wikipedia

In [ ]:
%pip uninstall -y arxiv
%pip install arxiv==1.4.8

In [ ]:
# tools (arxiv, wikipedia)
import arxiv
# Fix HTTP 301 error by using HTTPS for arXiv API queries
arxiv.Client.query_url_format = "https://export.arxiv.org/api/query?{}"

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

In [ ]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv, description="Query Arxiv papers")
print(arxiv.name)

In [ ]:
arxiv.invoke("Attention is all you need")

In [ ]:
wiki_wrapper_arxiv = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=5000)
wiki = WikipediaQueryRun(api_wrapper=wiki_wrapper_arxiv)
print(wiki.name)

In [ ]:
# wiki.invoke("Attention is all you need")
# wiki.invoke("What is the latest research on quantum computing?")
wiki.invoke("Research papers on machine learning")


In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os

os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

In [ ]:
tavily.invoke("provide me recent AI news")

In [ ]:
## combining all three tools [arxiv, wikipedia, tavily]

tools = [arxiv, wiki, tavily]

In [ ]:
## initilize LLM model
%pip install -U langchain-groq

In [ ]:
from langchain_groq import ChatGroq

# Note: qwen-qwq-32b was decommissioned by Groq. Using active model with tool calling support:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
llm.invoke("What is AI")

In [ ]:
llm_with_tools = llm.bind_tools(tools=tools)

In [ ]:
llm_with_tools.invoke("Give me the latest news on AI")

In [ ]:
llm_with_tools.invoke("Give me the research papers on quantum computing")

In [ ]:
llm_with_tools.invoke("what is machine learning?")

In [ ]:
## Workflow

from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage ## human message or I message
from typing import Annotated ## Labelling
from langgraph.graph.message import add_messages ## reducers in Langgraph

In [ ]:
class State(TypedDict):
    messages : Annotated[list[AnyMessage], add_messages]

In [ ]:
## Creating chatbot with Langgraph

from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition ## Tool Condition: when should we call tool node

In [ ]:
# ## node defination

# def tool_calling_llm(state: State):
#     return {"messages" : [llm_with_tools.invoke(state["messages"])]}

# ## building the graph
# ## nodes
# builder = StateGraph(State)
# builder.add_node("tool_calling_llm", tool_calling_llm)
# builder.add_node("tools", ToolNode(tools))

# ##edges
# builder.add_edge(START, "tool_calling_llm")
# builder.add_conditional_edges(
#     "tool_calling_llm", 
#     # If the latest message (result) from assistant IS a tool call -> tools_condition routes to tools
#     # If the latest message (result) from assistant IS NOT a tool call -> tools_condition routes to END
#     tools_condition
# )
# builder.add_edge("tools", END)

In [ ]:
# # compile

# graph = builder.compile()

In [ ]:
# display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# messages = graph.invoke({"messages" : "1706.03762"})
# for m in messages['messages']:
#     m.pretty_print()

In [ ]:
## node defination

def tool_calling_llm(state: State):
    return {"messages" : [llm_with_tools.invoke(state["messages"])]}

## building the graph
## nodes
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

##edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm", 
    # If the latest message (result) from assistant IS a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant IS NOT a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools", "tool_calling_llm")

In [ ]:
graph1 = builder.compile()

In [ ]:
display(Image(graph1.get_graph().draw_mermaid_png()))

In [ ]:
messages = graph1.invoke({"messages" : "tell me about deep learning and alsong with that give me some research papers related deep learning"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "who discovered gravity? answer in 5 wrod"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "which scientist discovered graveti? give me in 5 wrodds"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "which scientistss discover a gravity! find the grammatical errors and rectify it"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "hii my name is aarjav"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "hii my name is aarjav and give me research paper related machine learning"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "hii my name is aarjav"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "what are current news on AI"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "who discovered bulb"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "in which company i can invest my money in indian stock market"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "give me the names of top 2 indian companies with highest valuation"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "can u give me the symptoms of fever"})
for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = graph1.invoke({"messages" : "what is the leading cause of depression and tension"})
for m in messages['messages']:
    m.pretty_print()